# Amazon Bedrock End-to-End Demo: E-Bike Product Assistant

This notebook teaches common Amazon Bedrock application patterns through one connected example.

We will build a simple AI-powered e-bike experience that can:

- Generate a product description
- Generate synthetic customer feedback for testing
- Summarize many reviews
- Answer questions from supplied context
- Compare on-demand and batch-oriented patterns
- Build a conversational assistant
- Persist conversation memory in Amazon DynamoDB
- Clean up AWS resources created during the demo

> The notebook is designed for teaching. AWS calls are easy to identify, and most sections can first be explained without executing the live cells.


## 1. Learner AWS setup

Use the first code cell below for AWS sign-in: the default is your named AWS profile, SSO profile, or the runtime's attached role. Set `AWS_AUTH_METHOD = "keys"` only if your instructor has provided temporary credentials; the access key, secret, and optional session token are entered through hidden prompts and are not saved in this file.

Before the live cells, ask your AWS administrator to assign a learner execution role with `bedrock:InvokeModel` for the selected Nova inference profile, and DynamoDB `CreateTable`, `DescribeTable`, `PutItem`, `Query`, and `DeleteTable` for this demo's conversation table. Keep the model ARN and existing table permissions restricted to the class Region and table. See `setup/bedrock_course_roles.md` in the course repository for the role policy and profile setup.


In [ ]:
import getpass
import os
import boto3

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
AWS_AUTH_METHOD = "profile"  # Change to "keys" only when a profile is unavailable.

if AWS_AUTH_METHOD == "profile":
    AWS_PROFILE_NAME = os.getenv("AWS_PROFILE") or None
    aws_session = boto3.Session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
    boto3.setup_default_session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
    print("Using AWS profile/default credential chain.")
elif AWS_AUTH_METHOD == "keys":
    print("Enter temporary credentials only. Inputs are hidden and are not saved in this notebook.")
    access_key_id = getpass.getpass("AWS access key ID: ")
    secret_access_key = getpass.getpass("AWS secret access key: ")
    session_token = getpass.getpass("AWS session token (leave blank if not applicable): ")
    credential_args = {
        "aws_access_key_id": access_key_id,
        "aws_secret_access_key": secret_access_key,
        "region_name": AWS_REGION,
    }
    if session_token:
        credential_args["aws_session_token"] = session_token
    aws_session = boto3.Session(**credential_args)
    boto3.setup_default_session(**credential_args)
    del access_key_id, secret_access_key, session_token, credential_args
    print("Using temporary credentials for this kernel session.")
else:
    raise ValueError('AWS_AUTH_METHOD must be "profile" or "keys".')

import json
import random
import time
import uuid
from decimal import Decimal
from botocore.exceptions import ClientError

REGION = AWS_REGION
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.amazon.nova-lite-v1:0")

bedrock_runtime = aws_session.client("bedrock-runtime", region_name=REGION)
dynamodb = aws_session.resource("dynamodb", region_name=REGION)

print("Region:", REGION)
print("Model:", MODEL_ID)


## 2. A reusable Bedrock text-generation function

For many text-generation tasks, the application follows the same pattern:

**Prompt → Bedrock Runtime → Foundation model → Generated response**

The function below uses the Amazon Bedrock `Converse` API so that we can reuse the same interface for generation, summarization, question answering, and conversation.


In [ ]:
def converse_text(
    prompt,
    system_prompt=None,
    max_tokens=500,
    temperature=0.7,
    top_p=0.9,
    model_id=MODEL_ID
):
    messages = [
        {
            "role": "user",
            "content": [{"text": prompt}]
        }
    ]

    kwargs = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "maxTokens": max_tokens,
            "temperature": temperature,
            "topP": top_p
        }
    }

    if system_prompt:
        kwargs["system"] = [{"text": system_prompt}]

    response = bedrock_runtime.converse(**kwargs)

    return response["output"]["message"]["content"][0]["text"]


# Part A — Text Generation

## 3. Generate an e-bike product description

Imagine we are building an e-commerce application for an electric bike.

Instead of writing every description manually, we can ask the foundation model to generate a customer-friendly product description from structured product information.


In [ ]:
ebike_product = {
    "name": "VoltTrail X1",
    "motor": "750W rear hub motor",
    "battery": "48V 15Ah lithium-ion battery",
    "range": "up to 70 km",
    "top_speed": "25 km/h",
    "frame": "lightweight aluminum",
    "brakes": "hydraulic disc brakes",
    "gears": "7-speed drivetrain",
    "use_case": "daily commuting and weekend trail rides"
}

description_prompt = f"""
Create an engaging product description for the following e-bike.

Product details:
{json.dumps(ebike_product, indent=2)}

Requirements:
- 2 short paragraphs
- Customer-friendly language
- Mention practical benefits, not only specifications
- Do not invent features that are not listed
"""

print(description_prompt)


In [ ]:
product_description = converse_text(
    description_prompt,
    system_prompt="You are a helpful product copywriter.",
    temperature=0.6
)

print(product_description)


## Teaching point: inference parameters

Model output can be influenced using inference parameters.

- **maxTokens** limits the response length.
- **temperature** affects randomness.
- **topP** limits token selection to a probability mass.
- Lower temperature values are usually more predictable.
- Higher temperature values can produce more varied language.

Try changing the temperature and comparing the generated descriptions.


In [ ]:
for temp in [0.1, 0.7, 1.0]:
    print("\n" + "=" * 70)
    print("TEMPERATURE:", temp)

    response = converse_text(
        description_prompt,
        system_prompt="You are a helpful product copywriter.",
        temperature=temp,
        max_tokens=300
    )

    print(response)


# Part B — Synthetic Feedback Generation

## 4. Generate test customer feedback

Before real customer reviews are available, developers may need realistic-looking **synthetic test data**.

This is useful for testing:

- sentiment analysis
- review summarization
- search
- dashboards
- recommendation systems

The generated reviews below are for software testing only.


In [ ]:
def build_review_prompt(product_name, sentiment):
    if sentiment == "positive":
        rating = random.choice([4, 5, 5])
        prompt = f"""
Write a realistic positive customer review for the {product_name} e-bike.

Mention:
- riding experience
- battery life
- comfort
- braking or handling

Length: 70-100 words.
This is synthetic test data.
"""

    elif sentiment == "neutral":
        rating = 3
        prompt = f"""
Write a balanced 3-star customer review for the {product_name} e-bike.

Mention:
- at least one strength
- at least one limitation
- realistic commuting experience

Length: 70-100 words.
This is synthetic test data.
"""

    else:
        rating = random.choice([1, 2])
        prompt = f"""
Write a realistic critical customer review for the {product_name} e-bike.

Mention:
- one or two practical problems
- something the customer still found useful
- no abusive or hateful language

Length: 70-100 words.
This is synthetic test data.
"""

    return prompt, rating


In [ ]:
sentiments = [
    "positive",
    "positive",
    "positive",
    "neutral",
    "neutral",
    "negative"
]

synthetic_reviews = []

for i, sentiment in enumerate(sentiments, start=1):
    prompt, rating = build_review_prompt(ebike_product["name"], sentiment)

    review = converse_text(
        prompt,
        system_prompt="Generate realistic but clearly synthetic product-review test data.",
        temperature=0.8,
        max_tokens=180
    )

    synthetic_reviews.append({
        "review_id": i,
        "rating": rating,
        "sentiment": sentiment,
        "review": review
    })

for item in synthetic_reviews:
    print("\nReview", item["review_id"])
    print("Rating:", item["rating"])
    print("Sentiment:", item["sentiment"])
    print(item["review"])


# Part C — Feedback Summarization

## 5. Summarize all customer feedback

When many reviews are available, users often do not want to read every review.

A summarization prompt can ask the model to identify:

- overall sentiment
- common positive themes
- common complaints
- recurring product issues
- practical recommendations


In [ ]:
all_reviews_text = "\n\n".join(
    f"Rating: {r['rating']}/5\nReview: {r['review']}"
    for r in synthetic_reviews
)

summary_prompt = f"""
Summarize the following customer feedback for the VoltTrail X1 e-bike.

Reviews:
{all_reviews_text}

Return:

1. Overall customer sentiment
2. Top 3 positive themes
3. Top 3 concerns
4. Any repeated product issue
5. One concise recommendation for the product team

Base the answer only on the supplied reviews.
"""

print(summary_prompt[:4000])


In [ ]:
feedback_summary = converse_text(
    summary_prompt,
    system_prompt="You summarize customer feedback accurately and do not invent facts.",
    temperature=0.3,
    max_tokens=500
)

print(feedback_summary)


## 6. Extractive vs. abstractive summarization

Two useful summarization concepts are:

### Extractive summarization
Important words, phrases, or sentences are selected directly from the source.

### Abstractive summarization
The model creates new wording that captures the meaning of the source.

Foundation models are especially useful for abstractive summarization because they can combine multiple customer comments into a more natural summary.


# Part D — Question Answering

## 7. Ask questions about the product using supplied context

Question answering becomes more reliable when the application gives the model the relevant context.

Here the model is instructed to answer only from the product information and generated feedback summary.


In [ ]:
qa_context = f"""
PRODUCT INFORMATION:
{json.dumps(ebike_product, indent=2)}

CUSTOMER FEEDBACK SUMMARY:
{feedback_summary}
"""

question = "Is this e-bike suitable for daily commuting, and what should a buyer know before purchasing it?"

qa_prompt = f"""
Use only the context below to answer the question.

CONTEXT:
{qa_context}

QUESTION:
{question}

If information is missing, clearly say that it is not available in the context.
"""

answer = converse_text(
    qa_prompt,
    system_prompt="Answer only from supplied context.",
    temperature=0.2
)

print(answer)


# Part E — On-Demand vs. Batch-Oriented Processing

## 8. Choosing an inference pattern

Different workloads need different inference approaches.

### On-demand
Best when the user expects an immediate answer.

Examples:
- chatbot
- product assistant
- question answering
- recommendation interface

### Batch
Best when many independent prompts can be processed asynchronously.

Examples:
- summarize thousands of customer reviews overnight
- generate embeddings for a document collection
- generate test data for many products
- scheduled report generation

The following cell creates a small JSONL batch-style manifest to show the idea.


In [ ]:
batch_records = []

for sentiment in ["positive", "neutral", "negative"]:
    prompt, rating = build_review_prompt(ebike_product["name"], sentiment)

    batch_records.append({
        "recordId": str(uuid.uuid4()),
        "modelInput": {
            "schemaVersion": "messages-v1",
            "messages": [
                {
                    "role": "user",
                    "content": [{"text": prompt}]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 200,
                "temperature": 0.7,
                "topP": 0.9
            }
        }
    })

batch_manifest_path = "ebike_batch_manifest.jsonl"

with open(batch_manifest_path, "w", encoding="utf-8") as f:
    for record in batch_records:
        f.write(json.dumps(record) + "\n")

print(open(batch_manifest_path, encoding="utf-8").read())


## Teaching point: hybrid applications

A real application can use both patterns.

For example:

- Use **batch processing** overnight to summarize thousands of reviews.
- Store the summaries.
- Use **on-demand inference** when a customer asks a question.
- Supply the stored summary as context to the model.

This combines efficient offline processing with a responsive user experience.


# Part F — Conversation with Memory

## 9. Why memory is needed

Foundation model APIs do not automatically remember every previous conversation across independent application requests.

To maintain context, the application must send previous messages back to the model.

A common architecture is:

**User → Application → Conversation store → Amazon Bedrock**

For this demo, we will use **Amazon DynamoDB** as the conversation history store.


## 10. Create the DynamoDB table

The table will store messages using:

- `user_id` as the partition key
- `timestamp` as the sort key
- role: `user` or `assistant`
- message text
- TTL for automatic expiration

The table is created only when you run the following cell.


In [ ]:
CONVERSATION_TABLE_NAME = "bedrock-ebike-conversation-demo"

def create_conversation_table():
    try:
        table = dynamodb.Table(CONVERSATION_TABLE_NAME)
        table.load()
        print(f"Table already exists: {CONVERSATION_TABLE_NAME}")
        return table

    except ClientError as e:
        if e.response["Error"]["Code"] != "ResourceNotFoundException":
            raise

    print(f"Creating table: {CONVERSATION_TABLE_NAME}")

    table = dynamodb.create_table(
        TableName=CONVERSATION_TABLE_NAME,
        KeySchema=[
            {"AttributeName": "user_id", "KeyType": "HASH"},
            {"AttributeName": "timestamp", "KeyType": "RANGE"}
        ],
        AttributeDefinitions=[
            {"AttributeName": "user_id", "AttributeType": "S"},
            {"AttributeName": "timestamp", "AttributeType": "N"}
        ],
        BillingMode="PAY_PER_REQUEST"
    )

    table.wait_until_exists()

    # Enable TTL on the 'ttl' attribute.
    client = boto3.client("dynamodb", region_name=REGION)
    try:
        client.update_time_to_live(
            TableName=CONVERSATION_TABLE_NAME,
            TimeToLiveSpecification={
                "Enabled": True,
                "AttributeName": "ttl"
            }
        )
    except ClientError as e:
        print("TTL configuration message:", e)

    print("Table ready.")
    return table

conversation_table = create_conversation_table()


## 11. Store conversation messages

A TTL is included so old demo messages can expire automatically.

For teaching, we use a 1-day TTL.


In [ ]:
def store_message(user_id, role, message, ttl_days=1):
    now = int(time.time() * 1000)
    expires = int(time.time()) + ttl_days * 24 * 60 * 60

    conversation_table.put_item(
        Item={
            "user_id": user_id,
            "timestamp": Decimal(str(now)),
            "role": role,
            "message": message,
            "ttl": expires
        }
    )


## 12. Load conversation history

Before each model call, the application retrieves the previous messages and sends them to Amazon Bedrock along with the newest user message.


In [ ]:
from boto3.dynamodb.conditions import Key

def get_conversation_history(user_id, max_messages=10):
    response = conversation_table.query(
        KeyConditionExpression=Key("user_id").eq(user_id),
        ScanIndexForward=True
    )

    items = response.get("Items", [])

    # Keep only the most recent messages for this teaching demo.
    items = items[-max_messages:]

    messages = []

    for item in items:
        messages.append({
            "role": item["role"],
            "content": [{"text": item["message"]}]
        })

    return messages


## 13. Converse with DynamoDB-backed memory

The assistant will now remember earlier turns because the application explicitly retrieves and resends the conversation history.


In [ ]:
def chat_with_memory(user_id, user_message):
    # Retrieve previous history
    messages = get_conversation_history(user_id)

    # Add current user message
    messages.append({
        "role": "user",
        "content": [{"text": user_message}]
    })

    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=messages,
        system=[
            {
                "text": (
                    "You are a helpful e-bike product assistant. "
                    "Use the conversation history to understand follow-up questions. "
                    "Do not claim features that have not been provided."
                )
            }
        ],
        inferenceConfig={
            "maxTokens": 350,
            "temperature": 0.5,
            "topP": 0.9
        }
    )

    assistant_message = response["output"]["message"]["content"][0]["text"]

    # Save both sides of the conversation.
    store_message(user_id, "user", user_message)
    store_message(user_id, "assistant", assistant_message)

    return assistant_message


In [ ]:
DEMO_USER = "student-demo-user"

response1 = chat_with_memory(
    DEMO_USER,
    "I am considering the VoltTrail X1 mainly for commuting. "
    "I prefer good range and reliable braking."
)

print("Assistant:")
print(response1)


In [ ]:
response2 = chat_with_memory(
    DEMO_USER,
    "What were the two things I said matter most to me?"
)

print("Assistant:")
print(response2)


In [ ]:
response3 = chat_with_memory(
    DEMO_USER,
    "Based on that, explain in two sentences why this bike may or may not fit me."
)

print("Assistant:")
print(response3)


## 14. Inspect the stored conversation

This helps learners see that the memory exists in the **application data store**, not inside the foundation model itself.


In [ ]:
history = get_conversation_history(DEMO_USER, max_messages=20)

for i, message in enumerate(history, start=1):
    role = message["role"]
    text = message["content"][0]["text"]
    print(f"{i}. {role.upper()}: {text}\n")


## 15. Memory management considerations

As conversation history grows, sending every message forever becomes inefficient.

Applications commonly manage history by:

- keeping only the most recent N messages
- using a TTL
- summarizing older conversation history
- storing important facts separately
- using different conversation sessions
- combining short-term and long-term memory

The important design idea is:

**Conversation memory is an application responsibility.**


# Part G — Cleanup

## 16. Cleanup AWS resources created by this notebook

This notebook creates only one AWS resource:

- DynamoDB table: `bedrock-ebike-conversation-demo`

The following function deletes that table.

The Bedrock model itself is managed by AWS and is not created by this notebook, so there is nothing to delete for the model.


In [ ]:
def cleanup_demo_resources():
    print("Starting cleanup...")

    # Delete the DynamoDB table created by this notebook.
    try:
        table = dynamodb.Table(CONVERSATION_TABLE_NAME)
        table.load()

        print(f"Deleting DynamoDB table: {CONVERSATION_TABLE_NAME}")
        table.delete()
        table.wait_until_not_exists()

        print("DynamoDB table deleted.")

    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print("DynamoDB table does not exist. Nothing to delete.")
        else:
            raise

    print("Cleanup complete.")

# Run this only when you are finished with the demo:
# cleanup_demo_resources()


# Final Teaching Challenge

Ask learners to extend the application.

Suggested challenge:

1. Generate 10 synthetic e-bike reviews.
2. Use different positive, neutral, and negative ratios.
3. Summarize the feedback.
4. Ask the model a question using only the summary as context.
5. Start a conversation with memory.
6. Ask a follow-up question that depends on an earlier turn.
7. Inspect the DynamoDB history.
8. Run the cleanup function.

This demonstrates a complete generative AI application flow:

**Generate → Analyze → Summarize → Answer → Converse → Remember → Clean up**
